# Introducción a RL multiobjetivo

Authores: Juan Diego, Jessica, Daniel Barrero.



La comunidad de MORL (Multiobjective Reinforcement Learning) cuenta con los siguientes recursos:

* MO-Gymnasium: https://mo-gymnasium.farama.org/
  * Una adaptación multiobjetivo de varios ambientes clásicos de Gymnasium

* MORL Baselines: https://lucasalegre.github.io/morl-baselines
  * Una base de algoritmos multiobjetivo que se han desarrollado a lo largo de los años, análoga a Stable-baselines para RL en general.

* Weights and Biases (https://wandb.ai) como herramienta de visualización y evaluación del agente en términos de métricas de aprendizaje.

## 1) **MO-Gymnasium**


### Instalamos MO-Gymnasium con el siguiente comando.

In [1]:
import sys
!pip install mo-gymnasium
!{sys.executable} -m pip install moviepy

### Importamos

In [2]:
import gymnasium as gym
import mo_gymnasium as mo_gym

/opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Ejemplo de ambiente: Mountain Car



In [3]:
env = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array")

/opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## Mountain Car:

* El estado tiene dos componentes: (coordenada_x, rapidez). Esto hace que el espacio de estados de Mountain Car sea continuo y 2-dimensional.
* Tres acciones discretas: (0 = acelerar a la izquierda, 1 = no acelerar, 2 = acelerar a la derecha).
* Penalizaciones: -1 por cada paso de tiempo, -1 por acelerar en cualquier dirección.
* Versión multiobjetivo: El vector de recompensas tiene tres componentes. Específicamente, recompensa = [paso_de_tiempo, acelerar_IZQ, acelerar_DER].

## Visualización usando wrappers de Gymnasium

In [4]:
import os, imageio
os.makedirs("videos/demo", exist_ok=True)
# Nota: gymnasium.wrappers.RecordVideo usa moviepy internamente (bug con fps=None en v1.0.3).
# Grabamos manualmente con imageio en la siguiente celda.

In [5]:
# Grabamos el episodio con imageio (reemplaza RecordVideo que usa moviepy con bug de fps)
env_record = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array")

## Ejemplo con una política aleatoria para Mountain Car

El video se guarda como mp4 en la carpeta videos/demo de este notebook.

In [6]:
frames = []
obs, info = env_record.reset()
done = False

while not done:
    frame = env_record.render()
    if frame is not None:
        frames.append(frame)
    obs, vec_reward, terminated, truncated, info = env_record.step(env_record.action_space.sample())
    done = terminated or truncated

env_record.close()

if frames:
    imageio.mimsave("videos/demo/mountaincar_random.gif", frames, fps=30, loop=0)
    print(f"Video guardado: videos/demo/mountaincar_random.gif ({len(frames)} frames)")

Video guardado: videos/demo/mountaincar_random.gif (200 frames)


## 2) Usando algoritmos de MORL-baselines

In [7]:
# pycddlib requiere la librería GMP. En macOS: brew install gmp
# Luego instalar con las rutas de compilación correctas:
!CFLAGS="-I/opt/homebrew/include" LDFLAGS="-L/opt/homebrew/lib" pip install "pycddlib==2.1.6" --quiet

### Instalamos MORL-baselines con este comando:

In [8]:
!pip install git+https://github.com/LucasAlegre/morl-baselines.git

  Cloning https://github.com/LucasAlegre/morl-baselines.git to /private/var/folders/p8/ktqmc5j17c54vwmg9k44hg2h0000gn/T/pip-req-build-npcv1hue
  Running command git clone --filter=blob:none --quiet https://github.com/LucasAlegre/morl-baselines.git /private/var/folders/p8/ktqmc5j17c54vwmg9k44hg2h0000gn/T/pip-req-build-npcv1hue
  Resolved https://github.com/LucasAlegre/morl-baselines.git to commit fd8a9c6b6029f383408899bf482fca99bc1c5e81
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Mountain Car con *Pareto Q-learning* (PQL)

In [9]:
# Mountain Car

import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import MORecordEpisodeStatistics

GAMMA = 0.99

env = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("mo-mountaincar-v0", render_mode="rgb_array") # environment used for evaluation

In [10]:
import numpy as np
from morl_baselines.multi_policy.pareto_q_learning.pql import PQL

# Nota: PQL es tabular — requiere espacio de estados discreto o Box con dtype entero.
# mo-mountaincar-v0 tiene obs continuas (float32), por lo que PQL no puede indexar su tabla Q.
try:
    agent = PQL(
        env=env,
        ref_point=np.array([0, -50]),
        gamma=GAMMA,
        log=False,  # log=True requiere wandb configurado
    )
    agent.train(total_timesteps=100000, eval_env=eval_env, ref_point=np.array([0, -50]))
except Exception as e:
    print(f"PQL requiere estados discretos. Mountain Car (continuo) no es compatible: {e}")
    print("Para ambientes continuos usar GPI-LS, GPI-PD o PCN.")

PQL requiere estados discretos. Mountain Car (continuo) no es compatible: PQL only supports discretizable observation spaces.
Para ambientes continuos usar GPI-LS, GPI-PD o PCN.


## Deep Sea Treasure con PQL

El problema de Deep Sea Treasure (DST) (https://mo-gymnasium.farama.org/environments/deep-sea-treasure/) consiste de un submarino que debe buscar tesoros en el fondo del mar.

* La nave percibe una recompensa de -1 por cada paso de tiempo, y obtiene una recompensa positiva según el valor del tesoro que encontró.

* En general, a mayor profundiddad puede encontrar tesoros más valiosos, luego esto describe el balance o *trade-off*.

* El episodio termina cuando el submarino encuentra un tesoro.

In [11]:
import gymnasium as gym
import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import MORecordEpisodeStatistics

GAMMA = 0.99

env = mo_gym.make("deep-sea-treasure-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("deep-sea-treasure-v0") # environment used for evaluation

/opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [12]:
import numpy as np
from morl_baselines.multi_policy.pareto_q_learning.pql import PQL

agent = PQL(
    env=env,
    ref_point=np.array([0, -50]),
    gamma=GAMMA,
    log=False,  # log=True requiere wandb configurado
)

agent.train(total_timesteps=100000, eval_env=eval_env, ref_point=np.array([0, -50]))

{(0.699999988079071, -1.0),
 (8.03681981306076, -2.9701),
 (11.046854115, -4.90099501),
 (13.180722091614, -6.793465209301),
 (14.074187108950262, -7.72553055720799),
 (14.85618993228868, -8.64827525163591),
 (17.373143823765123, -12.247897700103202),
 (17.813676097383638, -13.12541872310217),
 (19.07265374771985, -15.705680661607312),
 (19.777976783050544, -17.383137616441328)}

### Visualización del agente de DST

In [13]:
import numpy as np
import imageio
import mo_gymnasium as mo_gym

def make_gif(env, agent, weight, fullpath: str, fps: int = 10, length: int = 300):
    """Render an episode and save it as a gif con imageio."""
    assert "rgb_array" in env.metadata["render_modes"], "Environment does not have rgb_array rendering."

    frames = []
    state, info = env.reset()
    terminated, truncated = False, False
    while not (terminated or truncated) and len(frames) < length:
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        # PQL indexa su tabla Q con enteros: convertir obs [col, row] → índice plano
        state_int = int(np.ravel_multi_index(state, agent.env_shape))
        action = agent.select_action(state_int, agent.score_hypervolume)
        state, reward, terminated, truncated, info = env.step(action)
    env.close()

    gif_path = fullpath + ".gif"
    imageio.mimsave(gif_path, frames, fps=fps, loop=0)
    print("Saved gif at: " + gif_path)


env_dst_render = mo_gym.make("deep-sea-treasure-v0", render_mode='rgb_array')
sample_weight_dst = np.array([0.5, 0.5])
make_gif(env_dst_render, agent, weight=sample_weight_dst, fps=10, fullpath="./deep_sea_treasure_agent")

Saved gif at: ./deep_sea_treasure_agent.gif


### Exercise 1:

Solving Resource Gathering using GPI-LS / GPI-PD

- Resource Gathering environment (see https://mo-gymnasium.farama.org/environments/resource-gathering/)
- GPI-LS (see https://lucasalegre.github.io/morl-baselines/algos/multi_policy/mp_mo_q_learning/).

Tips:
- You need to set weight_selection_algo='gpi-ls' and use_gpi_policy=True in the MPMOQLearning constructor in order to use GPI-LS.
- To use GPI-PD (the model-based version of GPI-LS), set dyna=True and gpi_pd=True
- Because this environment has stochastic transitions, set num_eval_episodes_for_front=50 in the train() method in order to evaluate the value of the policies with more precision.
- Use 10 iterations of 10k steps:
 total_timesteps=100000, timesteps_per_iteration=10000
- Use epsilon-greedy exploration with
    initial_epsilon=1.0,
    final_epsilon=0.05,
    epsilon_decay_steps=100000
- Observe the metrics in "eval/" panel of weights and biases (e.g., "eval/eum" for expected utility)

In [14]:
import gymnasium as gym
import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import MORecordEpisodeStatistics  # corregido: utils → wrappers
import numpy as np

GAMMA = 0.9
ref_point = np.array([-1., -1., -2.])

env = mo_gym.make("resource-gathering-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)

eval_env = mo_gym.make("resource-gathering-v0")

env.unwrapped.pareto_front(GAMMA)  # known Pareto front

/opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(


[array([-0.08269753,  0.22876792,  0.22876792]),
 array([-0.1260441 ,  0.34867844,  0.        ]),
 array([0.        , 0.        , 0.34867844]),
 array([-0.04782969,  0.20589113,  0.20589113]),
 array([-0.04782969,  0.3138106 ,  0.        ]),
 array([0.        , 0.28242954, 0.        ])]

In [15]:
from morl_baselines.multi_policy.multi_policy_moqlearning.mp_mo_q_learning import MPMOQLearning

# Your code here:
agent = MPMOQLearning(
    env,
    initial_epsilon=1.0,
    final_epsilon=0.05,
    epsilon_decay_steps=100000,
    gamma=GAMMA,
    dyna=True,
    gpi_pd=True,
    weight_selection_algo='gpi-ls',
    use_gpi_policy=True,
    log=False  # log=True requiere wandb configurado
)

agent.train(total_timesteps=100000, timesteps_per_iteration=10000, eval_env=eval_env, num_eval_episodes_for_front=50, ref_point=ref_point)

CCS: [] CCS size: 0
Next weight: [1. 0. 0.]
Adding value: [0. 0. 0.] to CCS.
W_corner: [array([1., 0., 0.]), array([0., 1., 0.]), array([0., 0., 1.])] W_corner size: 3
CCS: [array([0., 0., 0.], dtype=float32)] CCS size: 1
Next weight: [1. 0. 0.]
Adding value: [0. 0. 0.] to CCS.
removed value [0. 0. 0.]
W_corner: [array([1., 0., 0.]), array([0., 1., 0.]), array([0., 0., 1.])] W_corner size: 3
CCS: [array([0., 0., 0.])] CCS size: 1
Next weight: [0. 1. 0.]
Adding value: [-0.1497  0.3826  0.    ] to CCS.
W_corner: [array([0.7188, 0.2812, 0.    ]), array([1., 0., 0.]), array([0., 1., 0.]), array([0., 0., 1.])] W_corner size: 4
CCS: [array([0., 0., 0.]), array([-0.1497,  0.3826,  0.    ])] CCS size: 2
Next weight: [0. 0. 1.]
Adding value: [0.     0.     0.3874] to CCS.
removed value [0. 0. 0.]
W_corner: [array([0.7188, 0.2812, 0.    ]), array([1., 0., 0.]), array([0., 0., 1.]), array([0., 1., 0.]), array([0.    , 0.5031, 0.4969])] W_corner size: 5
CCS: [array([-0.1497,  0.3826,  0.    ]), ar

In [16]:
env.unwrapped.pareto_front(0.9)

[array([-0.0827,  0.2288,  0.2288]),
 array([-0.126 ,  0.3487,  0.    ]),
 array([0.    , 0.    , 0.3487]),
 array([-0.0478,  0.2059,  0.2059]),
 array([-0.0478,  0.3138,  0.    ]),
 array([0.    , 0.2824, 0.    ])]

In [17]:
agent.linear_support.ccs

[array([-0.1497,  0.3826,  0.    ]),
 array([0.    , 0.    , 0.3874]),
 array([-0.0856,  0.2573,  0.2573]),
 array([0.    , 0.3138, 0.    ]),
 array([-0.0213,  0.3719,  0.    ])]

### Exercise 2:

Use your learned agent and visualize how the learned behaviours change depending on the utility!

Use the make_gif function of morl-baselines (https://lucasalegre.github.io/morl-baselines/features/misc/#morl_baselines.common.utils.make_gif).

How does the policy for the following linear weights differ?
* [0.9, 0.1, 0.0]
* [0.3, 0.7, 0.0]
* [0.0, 1.0, 0.0]

In [18]:
from morl_baselines.common.utils import make_gif

env2 = mo_gym.make("resource-gathering-v0", render_mode='rgb_array')  # you need to set the render_mode to render the gifs

# Your code here:
make_gif(env2, agent, weight=np.array([0.9, 0.1, 0.0]), fps=10, fullpath="./myagent1")
make_gif(env2, agent, weight=np.array([0.3, 0.7, 0.0]), fps=10, fullpath="./myagent2")
make_gif(env2, agent, weight=np.array([0.0, 1.0, 0.0]), fps=10, fullpath="./myagent3")

objc[27044]: Class SDLApplication is implemented in both /opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x10ba392c8) and /opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x323208890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[27044]: Class SDLAppDelegate is implemented in both /opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x10ba39318) and /opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x3232088e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[27044]: Class SDLTranslatorResponder is implemented in both /opt/miniconda3/envs/cardozoenv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x10ba39390) and /opt/miniconda3/envs/cardozoenv/l

MoviePy - Building file ./myagent1.gif with imageio.


Saved gif at: ./myagent1.gif
MoviePy - Building file ./myagent2.gif with imageio.


Saved gif at: ./myagent2.gif
MoviePy - Building file ./myagent3.gif with imageio.


Saved gif at: ./myagent3.gif


## Minecart

Next, we will play with continuous states and function approximation!

In [19]:
import gymnasium as gym
import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import MORecordEpisodeStatistics  # corregido: utils → wrappers

### Pareto Conditioned Network (PCN)

Let's solve the Minecart problem (https://mo-gymnasium.farama.org/environments/minecart-deterministic) using PCN (https://lucasalegre.github.io/morl-baselines/algos/multi_policy/pcn)!

In [20]:
from morl_baselines.multi_policy.pcn.pcn import PCN


GAMMA = 1.0

env = mo_gym.make("minecart-deterministic-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("minecart-deterministic-v0") # environment used for evaluation

agent = PCN(
    env,
    scaling_factor=np.array([1.0, 1.0, 0.1, 0.1]),
    log=False,  # log=True requiere wandb configurado
)

agent.train(5000,  # reducido para demo — usar 1000000 para resultados completos
            eval_env=eval_env,
            ref_point=np.array([-1,-1,-200]),
            max_return=np.array([1.5,1.5,0.0]),
            max_buffer_size=200,
 )

step 5856 	 return [  0.51    0.64  -15.114], ([ 0.2615  0.3923 12.2547]) 	 loss 1.790E+00 	 horizons 216.7


### GPI-Linear Support (GPI-LS)

### Exercise 3

Now try to solve the stochastic version with GPI-LS (https://lucasalegre.github.io/morl-baselines/algos/multi_policy/gpi_pd)!

In [21]:
from morl_baselines.multi_policy.gpi_pd.gpi_pd import GPILS

GAMMA = 0.98

env = mo_gym.make("minecart-v0")
env = MORecordEpisodeStatistics(env, gamma=GAMMA)  # wrapper for recording statistics

eval_env = mo_gym.make("minecart-v0") # environment used for evaluation

# Your code here:
agent = GPILS(
    env,
    per=True,
    initial_epsilon=1.0,
    final_epsilon=0.05,
    epsilon_decay_steps=2000,
    target_net_update_freq=200,
    gradient_updates=10,
    log=False  # log=True requiere wandb configurado
)

agent.train(total_timesteps=2000,   # reducido para demo — usar 200000 para resultados completos
            eval_env=eval_env,
            ref_point=np.array([-1,-1,-200])
)